In [ ]:
import scanpy as sc

In [ ]:
adata = sc.read_h5ad("/data7/mark/STG/dataset/snRNA/merge_SCH_new/CSRES_4v3_500_1000gene/TH_downsampled.h5ad")

In [ ]:
adata.obs["celltype.L2"].value_counts()

In [ ]:
#adata_subset = adata[adata.obs["celltype.L2"].isin(["TH Tll1_Thsd7b Glut"])]
# Select the second most abundant cell type
adata_subset = adata[adata.obs["celltype.L2"].isin(adata.obs["celltype.L2"].value_counts().head(2).index[1:2])]
# Keep only male
adata_subset = adata_subset[adata_subset.obs["sex"] == "M"]
# Remove samples whose name contains MW22B
#adata_subset = adata_subset[~adata_subset.obs["sample"].str.contains("MW22B")]

In [ ]:
# Explore the subset data structure
print(f"adata_subset shape: {adata_subset.shape}")
print(f"Obs columns: {list(adata_subset.obs.columns)}")
print(f"X type: {type(adata_subset.X)}")
print(f"X min/max: {adata_subset.X.min():.4f} / {adata_subset.X.max():.4f}")
print(f"Number of cells: {adata_subset.n_obs}")
print(f"Number of genes: {adata_subset.n_vars}")
print()
# Check grouping information
print("celltype.L2 value counts:")
print(adata_subset.obs["celltype.L2"].value_counts())
print()
# Check for batch information
for col in adata_subset.obs.columns:
    if "batch" in col.lower():
        print(f"Batch column: {col}")
        print(adata_subset.obs[col].value_counts())

In [ ]:
# Check columns that can be used for grouping
for col in ["status", "region", "sex", "donor", "date"]:
    if col in adata_subset.obs.columns:
        print(f"\n{col} value counts:")
        print(adata_subset.obs[col].value_counts().head(20))

In [ ]:
# Check whether adata has raw or layers
print("Has raw:", adata.raw is not None)
if hasattr(adata, 'layers'):
    print("Layers keys:", list(adata.layers.keys()))
print("adata_subset.X shape:", adata_subset.X.shape)
print("adata_subset.X type:", type(adata_subset.X))

# Check whether var has gene names
print("\nvar columns:", list(adata_subset.var.columns[:20]))
print("var index head:", adata_subset.var.index[:5].tolist())

In [ ]:
import sys
sys.path.insert(0, "/home/junyichen/code/RUVAEDEG")
from model import RUVVAE_DEG, ZINBLoss, compute_deg, compute_deg_all_methods_legacy

import torch
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

# Settings
batch_key = "company"
group_key = "status"
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# DEG test mode: "wald" (default, closest to the DESeq2 Wald idea),
# "welch", "permutation", "bayes" or "all".
DEG_METHOD = "welch"

In [ ]:
hkg_priority = [
    "Aars", "Sars",
    "Polr2a", "Polr2f",
    "Psmd6", "Psmd7", "Psma5",
    "Rer1", "Ipo8", "Pop4", "Pes1",
    "Oaz1",
    "Rpl13a", "Rpl27", "Rps13", "Rps20",
    "Hprt1", "Gusb", "Ppia", "Ywhaz"
]

candidate_negative_control_genes = [
    # Relatively stable across large-scale cross-tissue analyses
    "Oaz1",
    "Rps13",
    "Rps20",
    "Rpl27",

    # Housekeeping genes reported as relatively stable in mouse brain
    "Aars",
    "Polr2f",
    "Psmd6",
    "Psmd7",
    "Psma5",

    # RNA processing, nucleocytoplasmic transport and basic cell maintenance
    "Ipo8",
    "Pop4",
    "Pes1",
    "Rer1",

    # Can be used as additional candidates, but must be validated in this data
    "Rpl13a",
    "Cyc1",
    "Sdha",
    "Ubc"
]

hkg_priority = candidate_negative_control_genes 

In [ ]:
# ========== 1. Prepare data (all genes) [ZINB mode: use raw counts as input] ==========
adata_hvg = adata_subset.copy()

# Clean inf/nan from the data
X_data = adata_hvg.X.toarray() if hasattr(adata_hvg.X, 'toarray') else adata_hvg.X.copy()
X_data = np.nan_to_num(X_data, nan=0.0, posinf=0.0, neginf=0.0)
adata_hvg.X = X_data

# Normalizing to median total counts
sc.pp.normalize_total(adata_hvg)
# Logarithmize the data (for DEG analyses such as logFC)
sc.pp.log1p(adata_hvg)

# Use all genes (no HVG filtering)
n_genes_total = adata_hvg.n_vars
Y_all = X_data  # (n_cells, n_genes)
gene_names = adata_hvg.var.index.tolist()
print(f"Total number of genes: {n_genes_total}")
print(f"Expression matrix shape: {Y_all.shape}")
print(f"Expression matrix range: {Y_all.min():.4f} ~ {Y_all.max():.4f}")

# Group labels
group_labels = adata_hvg.obs[group_key].values
groups_unique = sorted(np.unique(group_labels).tolist())
print(f"Groups: {groups_unique}")
print(f"CON: {(group_labels==groups_unique[0]).sum()}, CSDS: {(group_labels==groups_unique[1]).sum()}")

# Batch labels
batch_labels = adata_hvg.obs[batch_key].values
batches_unique = sorted(np.unique(batch_labels).tolist())
print(f"Number of batches: {len(batches_unique)}")
print(f"Batches: {batches_unique}")

In [ ]:
# ========== 1b. Compute n_genes_on (number of detected genes per cell, used as a continuous covariate) ==========
# Compute the number of genes with expression > 0 per cell from raw counts.
x_raw = adata_hvg.layers["counts"].toarray() if hasattr(adata_hvg.layers["counts"], "toarray") else adata_hvg.layers["counts"]
x_raw = np.nan_to_num(x_raw, nan=0.0, posinf=0.0, neginf=0.0)
n_genes_on_raw = (x_raw > 0).sum(axis=1).astype(np.float32)

# Continuous covariates must be standardized, otherwise W_cov becomes unstable due to large scale.
n_genes_on_mean = float(n_genes_on_raw.mean())
n_genes_on_std = float(n_genes_on_raw.std())
if n_genes_on_std < 1e-8:
    raise ValueError("n_genes_on has insufficient variation to be used as a continuous covariate")
n_genes_on = ((n_genes_on_raw - n_genes_on_mean) / n_genes_on_std).astype(np.float32)
adata_hvg.obs["n_genes_on"] = n_genes_on
print(f"n_genes_on source: raw counts > 0")
print(f"raw mean/std: {n_genes_on_mean:.1f} / {n_genes_on_std:.1f}")
print(f"standardized range: {n_genes_on.min():.3f} ~ {n_genes_on.max():.3f}")

In [ ]:
# ========== 2. Select negative control genes (HKG literature pool + data-driven) ==========

# ---------- User option: NC selection strategy ----------
NC_MODE = "data_only"  # options: "data_only", "hkg_only", "hybrid"

# ---------- 2a. Define HKG candidate function (adapted to mouse gene names) ----------
import re

def build_candidate_nc_genes_mouse(gene_names):
    """Build the candidate negative-control gene pool (adapted to mouse gene symbol casing)."""
    
    # 1) Literature-curated housekeeping genes (Eisenberg & Levanon 2013, converted to mouse casing)
    eisenberg_hkg = {
        "Actb", "B2m", "Gapdh", "Hmbs", "Hprt1", "Pgk1", "Ppia",
        "Rpl13a", "Rplp0", "Sdha", "Tbp", "Tfrc", "Ubc", "Vcp",
        "Ywhaz", "C1orf43", "Chmp2a", "Emc7", "Gpi", "Psmb2",
        "Psmb4", "Rab7a", "Reep5", "Snrpd3", "Vps29",
        "Cltc", "Csnk2b", "Ddx5", "Eef1a1", "Eif4a2",
        "Hnrnpk", "Nono", "Rpl32", "Rps18", "Sf3b1",
        "Srsf3", "Sumo1", "Ube2d2", "Usp22", "Xrn2",
    }
    
    # 2) Ribosomal proteins (cytoplasmic + mitochondrial)
    rp_genes = {g for g in gene_names 
                if re.match(r'^(RP[SL]|Mrp[sl]|Rps|Rpl)[0-9]', g, re.IGNORECASE)}
    
    # 3) Translation elongation factors
    eef_genes = {g for g in gene_names if re.match(r'^(Eef[12]|EIF)', g, re.IGNORECASE)}
    
    # 4) Proteasome subunits
    psm_genes = {g for g in gene_names if re.match(r'^(Psm|PSM)[A-Da-d][0-9]', g)}
    
    # 5) Mitochondrial ribosomal proteins
    mrp_genes = {g for g in gene_names if re.match(r'^Mrp[sl][0-9]', g, re.IGNORECASE)}
    
    candidate = (eisenberg_hkg & set(gene_names)) | rp_genes | eef_genes | psm_genes | mrp_genes
    return np.array([g in candidate for g in gene_names])


# ---------- 2b. Compute the expression matrix ----------
counts = adata_hvg.layers["counts"].toarray()
counts = np.nan_to_num(counts, nan=0.0, posinf=0.0, neginf=0.0)
Y_log = np.log1p(counts)

# Compute group means by status
mask_con = group_labels == groups_unique[0]
mask_csds = group_labels == groups_unique[1]
mean_con = Y_log[mask_con].mean(0)
mean_csds = Y_log[mask_csds].mean(0)
fc_abs = np.abs(mean_csds - mean_con)  # absolute logFC between the two groups

# ---------- 2c. Build HKG + priority list ----------
hkg_mask = build_candidate_nc_genes_mouse(gene_names)
print(f"HKG candidate genes (literature + ribosomal + proteasome etc.): {hkg_mask.sum()}")

# User-defined priority HKG list (ensure they are always in the negative controls)
priority_mask = np.zeros(n_genes_total, dtype=bool)
for hkg in hkg_priority:
    if hkg in gene_names:
        idx = gene_names.index(hkg)
        priority_mask[idx] = True
print(f"Priority HKG list hits: {priority_mask.sum()} / {len(hkg_priority)}")

# Total HKG pool = literature candidates + priority list
hkg_combined_mask = hkg_mask | priority_mask

# ---------- 2d. Select negative control genes according to NC_MODE ----------
n_nc_total = 500  # total number of negative control genes

if NC_MODE == "data_only":
    # Purely data-driven: pick the n_nc_total genes with the smallest logFC from all genes
    nc_idx = np.argsort(fc_abs)[:n_nc_total]
    print(f"[mode: data_only] Picked {n_nc_total} genes with smallest |logFC| from all {n_genes_total} genes")

elif NC_MODE == "hkg_only":
    # Purely prior knowledge: use only the HKG literature pool + priority list
    available_hkg_idx = np.where(hkg_combined_mask)[0]
    if len(available_hkg_idx) >= n_nc_total:
        # More HKG than target: pick the ones with the smallest |logFC|
        hkg_fc = fc_abs.copy()
        hkg_fc[~hkg_combined_mask] = np.inf
        nc_idx = np.argsort(hkg_fc)[:n_nc_total]
        print(f"[mode: hkg_only] Picked {n_nc_total} from the HKG pool ({len(available_hkg_idx)})")
    else:
        # Not enough HKG: use all of them
        nc_idx = available_hkg_idx
        print(f"[mode: hkg_only] Only {len(available_hkg_idx)} HKG available, using all")

elif NC_MODE == "hybrid":
    # Hybrid: prior HKG + data-driven fill-in
    n_hkg = hkg_combined_mask.sum()
    if n_hkg >= n_nc_total:
        hkg_fc = fc_abs.copy()
        hkg_fc[~hkg_combined_mask] = np.inf
        nc_idx = np.argsort(hkg_fc)[:n_nc_total]
        print(f"[mode: hybrid] HKG={n_hkg} >= {n_nc_total}, picked the smallest |logFC| from HKG")
    else:
        remaining = n_nc_total - n_hkg
        non_hkg_fc = fc_abs.copy()
        non_hkg_fc[hkg_combined_mask] = np.inf
        data_idx = np.argsort(non_hkg_fc)[:remaining]
        nc_idx = np.concatenate([np.where(hkg_combined_mask)[0], data_idx])
        print(f"[mode: hybrid] Kept all HKG={n_hkg} + data-driven fill of {remaining}")
else:
    raise ValueError(f"NC_MODE = {NC_MODE} not supported; use 'data_only' / 'hkg_only' / 'hybrid'")

neg_control_mask = np.zeros(n_genes_total, dtype=bool)
neg_control_mask[nc_idx] = True

# ---------- 2e. Print detailed statistics ----------
nc_hkg_overlap = neg_control_mask & hkg_combined_mask
nc_priority_overlap = neg_control_mask & priority_mask
print(f"\nFinal negative control genes: {neg_control_mask.sum()}")
print(f"  of which HKG literature candidates: {nc_hkg_overlap.sum()}")
print(f"  of which priority HKG list: {nc_priority_overlap.sum()}")
print(f"  of which non-HKG (hybrid mode only): {neg_control_mask.sum() - nc_hkg_overlap.sum()}")
print(f"\nNegative control genes (first 10): {[gene_names[i] for i in nc_idx[:10]]}")
print(f"Negative control gene logFC range: {fc_abs[nc_idx].min():.4f} ~ {fc_abs[nc_idx].max():.4f}")

## 13. New model test: SCVIWithDiseaseEffect (inherits scVI, reconstruction split into scVI part x disease-group linear effect)

In [ ]:
# ========== 13a. Import the new model + prepare scVI-format data ==========
import sys
sys.path.insert(0, "/home/junyichen/code/RUVAEDEG")
from model_scvi_disease import SCVIWithDiseaseEffect, VAEWithDiseaseEffect

import torch
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy.sparse import csr_matrix

# Use the same data subset as the RUVVAE above, but register it via scVI's setup_anndata
# Note: scVI requires raw counts in X or a layer (ZINB uses counts)
adata_scvi = adata_subset.copy()

# Ensure X is raw counts (scVI ZINB needs counts, not log1p)
if "counts" in adata_scvi.layers:
    adata_scvi.X = adata_scvi.layers["counts"].copy()
else:
    # If there is no counts layer, use the current X (assumed to be counts)
    pass

# Clean inf/nan
X_scvi = adata_scvi.X.toarray() if hasattr(adata_scvi.X, "toarray") else adata_scvi.X.copy()
X_scvi = np.nan_to_num(X_scvi, nan=0.0, posinf=0.0, neginf=0.0)
adata_scvi.X = csr_matrix(X_scvi.astype(np.float32))

# Compute n_genes_on (continuous covariate, consistent with RUVVAE)
x_raw = X_scvi
n_genes_on_raw = (x_raw > 0).sum(axis=1).astype(np.float32)
n_genes_on_mean = float(n_genes_on_raw.mean())
n_genes_on_std = float(n_genes_on_raw.std())
n_genes_on_scvi = ((n_genes_on_raw - n_genes_on_mean) / n_genes_on_std).astype(np.float32)
adata_scvi.obs["n_genes_on"] = n_genes_on_scvi

print(f"adata_scvi shape: {adata_scvi.shape}")
print(f"X dtype: {adata_scvi.X.dtype}, range: [{adata_scvi.X.min()}, {adata_scvi.X.max()}]")
print(f"status groups: {adata_scvi.obs[group_key].value_counts().to_dict()}")
print(f"batch groups: {adata_scvi.obs[batch_key].value_counts().to_dict()}")

In [ ]:
# ========== 13b. setup_anndata + instantiation + forward validation ==========
torch.manual_seed(seed)
np.random.seed(seed)

# Register: batch_key for technical batches, labels_key for disease groups (linear effect)
SCVIWithDiseaseEffect.setup_anndata(
    adata_scvi,
    batch_key=batch_key,          # "company"
    labels_key=group_key,         # "status" -> disease-group linear effect
    continuous_covariate_keys=["n_genes_on"],
)

model_disease = SCVIWithDiseaseEffect(
    adata_scvi,
    n_latent=32,
    n_layers=2,
    gene_likelihood="zinb",
    group_effect_scale=0.01,      # W_group init std
    group_effect_prior=0.0,       # L2 penalty weight (0 = no penalty)
)

print(f"Module type: {type(model_disease.module).__name__}")
print(f"W_group shape (n_labels, n_genes): {tuple(model_disease.module.W_group.shape)}")
print(f"Number of labels (status) categories: {model_disease.module.n_labels}")
print(f"Number of batches: {model_disease.module.n_batch}")

# ---- Forward validation: reconstruction split mu = mu_scvi * mu_disease ----
scdl = model_disease._make_data_loader(adata=adata_scvi, batch_size=256)
for tensors in scdl:
    inf, gen, loss = model_disease.module.forward(tensors, compute_loss=True)
    px = gen["px"]
    print(f"\npx.mu shape: {tuple(px.mu.shape)}")
    print(f"mu_scvi shape: {tuple(gen['mu_scvi'].shape)}")
    print(f"mu_disease shape: {tuple(gen['mu_disease'].shape)}")
    print(f"group_effect shape: {tuple(gen['group_effect'].shape)}")
    print(f"loss: {float(loss.loss):.4f}")

    # Key assertion: combined mean == scVI part x disease part
    torch.testing.assert_close(
        px.mu, gen["mu_scvi"] * gen["mu_disease"], rtol=1e-4, atol=1e-4
    )
    print("✓ Reconstruction split validated: mu == mu_scvi * mu_disease")

    # Disease effect should be positive (exp form)
    assert (gen["mu_disease"] > 0).all()
    print(f"✓ mu_disease > 0 (exp form), range: [{gen['mu_disease'].min():.4f}, {gen['mu_disease'].max():.4f}]")
    break

In [ ]:
# ========== 13c. Training + reconstruction split + disease logFC ==========
# Short training (for demonstration; increase max_epochs for real use)
model_disease.train(
    max_epochs=100,
    train_size=0.9,
    batch_size=256,
    early_stopping=True,
    early_stopping_patience=5,
    plan_kwargs={"lr": 1e-3},
)
print("Training complete")

# ---- Two reconstruction parts ----
mu_scvi, mu_disease = model_disease.get_reconstruction_parts(adata=adata_scvi, batch_size=256)
print(f"\nmu_scvi shape: {mu_scvi.shape}")
print(f"mu_disease shape: {mu_disease.shape}")
print(f"mu_scvi range: [{mu_scvi.min():.2f}, {mu_scvi.max():.2f}]")
print(f"mu_disease range: [{mu_disease.min():.4f}, {mu_disease.max():.4f}]")

# ---- Combined reconstruction ----
recon = model_disease.get_reconstruction(adata=adata_scvi, batch_size=256)
print(f"recon (mu_scvi * mu_disease) shape: {recon.shape}")
print(f"recon range: [{recon.min():.2f}, {recon.max():.2f}]")

# ---- Disease logFC (W_group[disease] - W_group[control]) ----
lfc = model_disease.get_disease_logfc()
print(f"\nDisease logFC shape: {lfc.shape}")
print(f"logFC range: [{lfc.min():.4f}, {lfc.max():.4f}]")
print(f"|logFC| mean: {np.abs(lfc).mean():.4f}")

# Show top disease-effect genes
gene_names_scvi = adata_scvi.var_names.tolist()
df_lfc = pd.DataFrame({"gene": gene_names_scvi, "disease_logFC": lfc})
df_lfc = df_lfc.assign(abs_lfc=np.abs(df_lfc["disease_logFC"])).sort_values("abs_lfc", ascending=False)
print("\nTop 10 disease-effect genes:")
display(df_lfc.head(10))

In [ ]:
# ========== 13d. Visualization: reconstruction split + disease effect ==========
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1) scVI part vs combined reconstruction
ax = axes[0]
idx_plot = np.random.choice(mu_scvi.shape[0], min(300, mu_scvi.shape[0]), replace=False)
ax.scatter(mu_scvi[idx_plot, :].ravel(), recon[idx_plot, :].ravel(), s=2, alpha=0.3)
ax.plot([0, recon.max()], [0, recon.max()], "r--", alpha=0.5)
ax.set_xlabel("mu_scvi (scVI part)")
ax.set_ylabel("recon = mu_scvi * mu_disease")
ax.set_title("scVI part vs combined reconstruction")
ax.grid(True, alpha=0.3)

# 2) Disease effect distribution (grouped by status)
ax = axes[1]
status_labels = adata_scvi.obs[group_key].values
for i, s in enumerate(sorted(np.unique(status_labels))):
    mask = status_labels == s
    ax.hist(np.log(mu_disease[mask]).mean(0), bins=50, alpha=0.5, label=s)
ax.set_xlabel("mean log(mu_disease) per gene")
ax.set_ylabel("count")
ax.set_title("Disease-group linear effect (log scale)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 3) Disease logFC distribution
ax = axes[2]
ax.hist(lfc, bins=80, color="steelblue", alpha=0.7)
ax.axvline(0, c="gray", ls="--", alpha=0.5)
ax.set_xlabel("disease logFC (W_group[disease] - W_group[control])")
ax.set_ylabel("count")
ax.set_title("Disease logFC distribution")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("=== [Cell 13] SCVIWithDiseaseEffect test complete ===")
print("✓ Fully inherits scvi.model.SCVI")
print("✓ Reconstruction split into mu_scvi (scVI) x mu_disease (disease linear effect)")
print("✓ Disease logFC = W_group[disease] - W_group[control]")

In [ ]:
df_lfc